# Getting Started with behav_utils

A walkthrough of the basic workflow: load → filter → stats → plot → group → compare.

Everything here uses `behav_utils` only — no project-specific code.

## 1. Load data

In [ ]:
%matplotlib inline
from shared_setup import *

In [ ]:
from behav_utils.data.selection import select_sessions
from behav_utils.plotting.styles import apply_style
apply_style()

import matplotlib.pyplot as plt
import numpy as np

# Load from YAML config (maps CSV columns to internal names)
experiment, info = load_data()

print(f"Animals: {experiment.n_animals}")
for aid, animal in list(experiment.animals.items())[:5]:
    print(f"  {aid}: {animal.n_sessions} sessions")

In [ ]:
# Pick one animal to explore
animal = list(experiment.animals.values())[0]
aid = list(experiment.animals.keys())[0]
print(f"Working with {aid}: {animal.n_sessions} sessions")

# Sessions are ordered chronologically
sess = animal.sessions[4]
print(f"\nFirst session: {sess.session_id}")
print(f"  Trials: {sess.n_trials}")
print(f"  Stage: {sess.metadata.get('stage')}")
print(f"  Date: {sess.date}")

In [ ]:
# Trial-level data is in numpy arrays
print(f"Stimulus range: [{sess.trials.stimulus.min():.2f}, {sess.trials.stimulus.max():.2f}]")
print(f"Choice values: {np.unique(sess.trials.choice[~np.isnan(sess.trials.choice.astype(float))])}")
print(f"Accuracy: {np.nanmean(sess.trials.correct):.2%}")

## 2. Filter sessions

`select_sessions` uses presets defined in `config.yaml` or custom `SessionFilter` objects.

In [ ]:
from behav_utils.data.selection import list_presets

print("Available presets:")
for name, filt in list_presets().items():
    print(f"  {name:20s}  {filt}")

In [ ]:
# Use a preset: expert_uniform picks expert-level sessions on uniform distribution
expert = select_sessions(animal, preset='expert_uniform')
print(f"Expert sessions: {len(expert)} / {animal.n_sessions}")

# Use a different preset
all_uniform = select_sessions(animal, preset='all_uniform')
print(f"All uniform sessions: {len(all_uniform)}")

# Custom filter
from behav_utils.data.selection import SessionFilter

custom = SessionFilter(stage='Full_Task_Cont', min_accuracy=0.60, first_n=10)
custom_sessions = custom.apply(animal)

# Or directly with the select_sessions function
custom_sessions = select_sessions(animal, stage='Full_Task_Cont', min_accuracy=0.60, first_n=10)

print(f"Custom filter: {len(custom_sessions)} sessions")

## 3. Plot psychometric curves

Three levels: single session, single animal (pooled), and grouped comparison.

In [ ]:
# Single session
fig, ax = plt.subplots(figsize=(5, 4))
expert[-1].plot_psychometric(ax=ax, n_bootstrap=100, show_ci=True)
ax.set_title(f'{aid} — last expert session')
custom_sessions[-1].plot_psychometric(ax=ax, n_bootstrap=100, show_ci=True, color='orange')
plt.show()

In [ ]:
# Pooled across sessions (one animal)
fig = animal.plot_psychometric()
plt.show()

## 4. Compute summary statistics

25+ registered stats available via `session.stats()`.

In [ ]:
from behav_utils.analysis.summary_stats import list_available_stats

print("Available stats:")
print(', '.join(sorted(list_available_stats())))

In [ ]:
# Compute a few stats for one session
stats = expert[-1].stats(['accuracy', 'pse', 'slope', 'recency', 'win_stay', 'side_bias'])
print(f"Stats for {expert[-1].session_id}:")
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k:>20s}: {v:.4f}")
    else:
        print(f"  {k:>20s}: {v}")

In [ ]:
# Compute across all expert sessions → trajectory
from behav_utils.analysis.summary_stats import compute_summary_stats

stat_names = ['accuracy', 'pse', 'slope']
results = []
for s in expert:
    r = s.stats(stat_names)
    r['session_id'] = s.session_id
    results.append(r)

print(f"{'Session':>20s}" + ''.join(f"{sn:>12s}" for sn in stat_names))
print('-' * (20 + 12 * len(stat_names)))
for r in results[-5:]:
    vals = ''.join(f"{r[sn]:>12.4f}" for sn in stat_names)
    print(f"{r['session_id']:>20s}{vals}")

## 5. Plot summary stats across sessions

Trajectory plots show how stats evolve over time.

In [ ]:
from behav_utils.plotting.trajectory import plot_stat_trajectory, plot_multi_animal_trajectory

# Single animal trajectory
fig, ax = plt.subplots(figsize=(6, 4))
animal.plot_trajectory('accuracy', ax=ax)

In [ ]:
# Multi-animal trajectory (cohort mean ± SEM)
all_animals = list(experiment.animals.values())
fig, ax = plot_multi_animal_trajectory(
    all_animals, 'accuracy', combine='mean_sem',
    title='Cohort accuracy trajectory')
plt.show()

In [ ]:
# Grid: multiple stats for one or more animals
from behav_utils.plotting.trajectory import plot_stat_grid

fig, axes = plot_stat_grid(
    all_animals[:4],
    stats=['accuracy', 'pse', 'recency', 'win_stay'],
)
plt.show()

## 6. Group data and compare

Group sessions into named sets, then compare with overlaid or side-by-side psychometrics.

The key data structure is `Dict[str, List[SessionData]]` — labels map to session lists.

In [ ]:
# Group by learning stage: first 5 vs last 5 sessions
groups = {
    'Early (first 5)': all_uniform[:5],
    'Late (last 5)':   all_uniform[-5:],
}
print(f"Groups: {', '.join(f'{k}: {len(v)} sessions' for k, v in groups.items())}")

In [ ]:
from behav_utils.plotting.psychometric import plot_psychometric_overlay

# Overlay on one axes — direct visual comparison
fig, ax, infos = plot_psychometric_overlay(
    groups,
    mode='pooled',          # pool trials per group
    n_bootstrap=200,        # bootstrap CI bands
    title=f'{aid} — early vs late',
)
plt.show()

# Print fitted parameters
for label, info in infos.items():
    print(f"  {label}: PSE={info.get('mu', 0):.3f}, "
          f"slope={info.get('sigma', 0):.3f}")

In [ ]:
from behav_utils.plotting.psychometric import plot_psychometric_compare

# Side-by-side subplots — one panel per group
fig, axes, infos = plot_psychometric_compare(
    groups,
    mode='session_mean',   # per-session mean ± SEM
    suptitle=f'{aid} — early vs late',
)
plt.show()

## 7. Group across animals

Same pattern works for grouping different animals' sessions.

In [ ]:
# Group by animal (first 3 animals, expert sessions only)
animal_groups = {}
for a_id, a in list(experiment.animals.items())[:3]:
    try:
        a_expert = select_sessions(a, preset='expert_uniform')
        if a_expert:
            animal_groups[a_id] = a_expert
    except ValueError:
        pass  # Skip animals with no expert sessions

print(f"Animals with expert data: {len(animal_groups)}")

if animal_groups:
    fig, ax, infos = plot_psychometric_overlay(
        animal_groups,
        mode='pooled',
        n_bootstrap=200,
        title='Expert psychometrics — per animal',
    )
    plt.show()

## 8. Update matrices

The update matrix shows how the previous stimulus influences the current choice.

In [ ]:
from behav_utils.analysis.update_matrix import compute_update_matrix, compute_update_matrix_from_sessions
from behav_utils.plotting.update_matrix import plot_update_matrix, plot_phase_update_matrices

# Single session
stim = expert[-1].trials.stimulus
ch = expert[-1].trials.choice.astype(float)
cat = expert[-1].trials.category
valid = ~np.isnan(ch)
um, cond, info = compute_update_matrix(stim[valid], ch[valid], cat[valid])

fig, ax = plt.subplots(figsize=(5, 4))
plot_update_matrix(um, ax=ax, title=f'{aid} — expert UM')
plt.show()

In [ ]:
# Pooled across sessions
um_pooled = compute_update_matrix_from_sessions(expert, method='pool')

# Compare phases
if len(all_uniform) > 10:
    phases = {
        'Early': compute_update_matrix_from_sessions(all_uniform[:5], method='pool')[0],
        'Late':  compute_update_matrix_from_sessions(all_uniform[-5:], method='pool')[0],
    }
    fig, axes = plot_phase_update_matrices(phases)
    plt.show()

## Summary

All of the above uses `behav_utils` only. The key patterns:

| Task | Function | Input |
|:-----|:---------|:------|
| Load | `load_experiment(config)` | YAML path |
| Filter | `select_sessions(animal, preset=)` | AnimalData + preset name |
| Psychometric | `session.plot_psychometric()` | SessionData |
| Stats | `session.stats(stat_names)` | List of stat names |
| Trajectory | `plot_stat_trajectory(animal, stat)` | AnimalData + stat name |
| Group compare | `plot_psychometric_overlay(groups)` | `Dict[str, List[SessionData]]` |
| Update matrix | `compute_update_matrix_from_sessions(sessions)` | List of SessionData |

For project-specific analysis (models, SBI, opto, adaptation), see the
`analysis/`, `inference/`, and `plotting/` modules in the project repo.